# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema and can be loaded from:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Keywords: {metadata.keywords}")
print(f"Date Published: {metadata.datePublished}")
print(f"Data Collection Type: {metadata.dataCollectionType}")

## 2. Data Overview
Review available record sets, their fields, and respective `@id` values.

The FAIR^2 dataset contains several tables (record sets):
- Table 1: Clinical features
- Table 2: Hormone levels, imaging, ultrasound
- Table 3: CYP21A2 genetic variants

We'll enumerate available record sets and their fields using their `@id` values.

In [ ]:
# List record sets in the dataset
record_sets = dataset.list_record_sets()

print("Available record sets and fields:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs['name']}")
    print("Fields:")
    for fld in dataset.list_fields(rs['@id']):
        print(f"   Field @id: {fld['@id']}, name: {fld['name']}, dataType: {fld.get('dataType','')}")
    print("")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis, referencing by `@id`.

We'll extract records for each record set and preview them.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nData preview for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Apply processing steps such as:
- Filtering records (e.g., age > threshold)
- Normalizing numeric columns
- Grouping (e.g., by phenotype or diagnosis)

**Note:** All operations reference fields by their `@id`.

Let's select Table 1 (clinical features) for demonstration.

In [ ]:
# Identify record set with clinical features (Table 1)
table1_rs = None
for rs in dataset.list_record_sets():
    if 'clinical' in rs['name'].lower() or 'feature' in rs['name'].lower() or 'Table 1' in rs['name']:
        table1_rs = rs['@id']
        break
if table1_rs is None:
    table1_rs = record_set_ids[0]  # fallback

# Find the field @id for age at diagnosis (numeric analysis)
age_field_id = None
for fld in dataset.list_fields(table1_rs):
    if ('age' in fld['name'].lower() and 'diagnosis' in fld['name'].lower()) or 'age' in fld['name'].lower():
        age_field_id = fld['@id']
        break
if age_field_id is None:
    age_field_id = dataframes[table1_rs].columns[0]  # fallback if not found

df_clinical = dataframes[table1_rs]
# Filter: age at diagnosis > 20
threshold = 20
filtered_df = df_clinical[df_clinical[age_field_id] > threshold]
print(f"Filtered records in Table 1 (clinical features) where {age_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
print(f"Normalized '{age_field_id}' for filtered records:")
display(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

# Grouping by phenotype, if present
group_field = None
for fld in dataset.list_fields(table1_rs):
    if 'phenotype' in fld['name'].lower():
        group_field = fld['@id']
        break
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[age_field_id].mean().reset_index()
    print(f"Grouped mean age at diagnosis by phenotype ({group_field}):")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of age at diagnosis and the relationship to phenotypes (if available).

In [ ]:
# Plot histogram for age at diagnosis
plt.figure(figsize=(6, 4))
df_clinical[age_field_id].hist(bins=10)
plt.xlabel('Age at Diagnosis')
plt.ylabel('Count')
plt.title('Distribution of Age at Diagnosis')
plt.show()

# If phenotype grouping exists, plot mean age by phenotype
if group_field and group_field in df_clinical.columns:
    grouped = df_clinical.groupby(group_field)[age_field_id].mean().reset_index()
    plt.figure(figsize=(6, 4))
    plt.bar(grouped[group_field], grouped[age_field_id])
    plt.xlabel('Phenotype')
    plt.ylabel('Mean Age at Diagnosis')
    plt.title('Mean Age at Diagnosis by Phenotype')
    plt.show()

## 6. Conclusion

This notebook demonstrated the loading, overview, extraction, processing, and visualization of the FAIR^2 dataset using `mlcroissant`, referencing all entities and fields by their `@id`. The clinical features, hormone levels, imaging, and genetic variants were accessed, and basic EDA was performed on age-related clinical information.

Remember to reference all fields and record sets using their respective `@id`s for reliable and reproducible analysis with Croissant datasets.